# 03. QA Dataset Audit
This notebook validates the final human-reviewed QA dataset against the stratified target distribution and checks for structural anomalies before running experiments.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/qa_dataset/qa_pairs_full.jsonl")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"File not found: {DATA_PATH}")

data = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

df = pd.DataFrame(data)
print(f"Total QA Pairs: {len(df)}")

## 1. Question Type Distribution
Target: ~30% factual_extractive, ~35% multi_section_reasoning, ~20% structural, ~15% boundary.

In [ ]:
type_counts = df['question_type'].value_counts()
type_pct = df['question_type'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Count': type_counts, 'Percentage (%)': type_pct})
display(dist_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=type_counts.values, y=type_counts.index, hue=type_counts.index, palette="viridis", legend=False)
plt.title("Question Type Distribution")
plt.xlabel("Number of Pairs")
plt.ylabel("Question Type")
plt.show()

## 2. Pairs per Verdict
Target: ~7 pairs per verdict. Flag any verdict with < 4 or > 12 pairs.

In [ ]:
verdict_counts = df['verdict_id'].value_counts()

plt.figure(figsize=(10, 5))
sns.histplot(verdict_counts, bins=15, kde=True)
plt.title("Distribution of QA Pairs per Verdict")
plt.xlabel("Pairs per Verdict")
plt.ylabel("Frequency")
plt.show()

outliers = verdict_counts[(verdict_counts < 4) | (verdict_counts > 12)]
if not outliers.empty:
    print("\n⚠️ OUTLIERS DETECTED (< 4 or > 12 pairs):")
    display(outliers)
else:
    print("\n✅ All verdicts fall within the acceptable volume range (4-12 pairs).")

## 3. Gold Paragraph Integrity
Checking for empty `gold_paragraphs` lists, empty strings, and evaluating average text lengths.

In [ ]:
empty_paragraphs = df[~df['gold_paragraphs'].astype(bool)]
print(f"Pairs with empty gold_paragraphs array: {len(empty_paragraphs)}")

df['gold_paragraphs_char_len'] = df['gold_paragraphs'].apply(lambda x: sum(len(p) for p in x) if isinstance(x, list) else 0)

plt.figure(figsize=(10, 5))
sns.histplot(df['gold_paragraphs_char_len'], bins=30, kde=True, color='coral')
plt.title("Length Distribution of Gold Paragraphs (Characters)")
plt.xlabel("Total Character Length")
plt.ylabel("Frequency")
plt.show()

if not empty_paragraphs.empty:
    display(empty_paragraphs[['question_id', 'verdict_id', 'question_type']])

## 4. IAA Summary
Runs the `iaa_calculator` module programmatically to output the current annotation agreement kappa status.

In [ ]:
import subprocess
import os
import sys

# Set project root for subprocess
project_root = os.path.abspath("..")
env = os.environ.copy()
env["PYTHONPATH"] = project_root + os.pathsep + env.get("PYTHONPATH", "")

try:
    result = subprocess.run(
        [sys.executable, "-m", "src.qa.iaa_calculator", "--annotator2", "../data/qa_dataset/qa_pairs_annotator2.jsonl"],
        capture_output=True,
        text=True,
        env=env
    )
    print(result.stdout)
    if result.stderr:
        print("Errors:\n", result.stderr)
except Exception as e:
    print(f"Could not run iaa_calculator: {e}")